## Traffic Incident System Dataset
https://opendata.transport.vic.gov.au/dataset/victoria-road-crash-data/resource/5df1f373-0c90-48f5-80e1-7b2a35507134

https://www.police.vic.gov.au/road-policing-statistics


In [0]:
%pip install osmnx folium
%restart_python

In [0]:
import requests

url = "https://opendata.transport.vic.gov.au/dataset/bb77800e-1857-4edc-bf9e-e188437a1c8e/resource/5df1f373-0c90-48f5-80e1-7b2a35507134/download/victorian_road_crash_data.csv"
response = requests.get(url)

with open('/Volumes/danny_catalog/dtp_schema/source/victorian_road_crash_data.csv', 'wb') as f:
    f.write(response.content)

In [0]:
(
  spark.read.csv('/Volumes/danny_catalog/dtp_schema/source/victorian_road_crash_data.csv', header=True, inferSchema=True)
  .write.mode('overwrite')
  .saveAsTable('danny_catalog.dtp_schema.victorian_road_crash_data')
)

In [0]:
%sql
SELECT * FROM danny_catalog.dtp_schema.victorian_road_crash_data

In [0]:
%sql
SELECT *, st_astext(ST_Buffer(st_point(longitude, latitude), 0.01)) as buffered_wkt 
FROM danny_catalog.dtp_schema.victorian_road_crash_data
WHERE LGA_NAME = "SHEPPARTON"


In [0]:
# 0.1 buffer is around 11.1 km in radius at the equator
pdf_buffered_accident_spot = spark.sql("""
                                        SELECT *, st_astext(ST_Buffer(st_point(longitude, latitude), 1)) as buffered_wkt 
                                        FROM danny_catalog.dtp_schema.victorian_road_crash_data
                                        WHERE LGA_NAME = "SHEPPARTON"
                                       """).toPandas()

In [0]:
import osmnx
import shapely.geometry
import shapely.wkt

polygon_geom = shapely.wkt.loads(pdf_buffered_accident_spot.iloc[0]['buffered_wkt'])
police_hospital_pdf = osmnx.features.features_from_polygon(polygon_geom, {'amenity': ['police', 'hospital']})

# Cast police_hospital_pdf geometry to string
police_hospital_pdf['geometry'] = police_hospital_pdf['geometry'].apply(lambda geom: geom.wkt)

In [0]:
df_first_accident_record = spark.createDataFrame([pdf_buffered_accident_spot.iloc[0].to_dict()])
from pyspark.sql.functions import expr
df_first_accident_record = df_first_accident_record.withColumn("centroid_accident", expr("st_astext(st_centroid(st_geomfromtext(buffered_wkt)))"))

display(df_first_accident_record)

In [0]:
from pyspark.sql.functions import expr

police_df = spark.createDataFrame(police_pdf)
police_df = police_df.withColumn("centroid_police_station", expr("st_astext(st_centroid(st_geomfromtext(geometry)))"))
display(police_df)

In [0]:
import osmnx as ox
from shapely.geometry import LineString, Point

ox.settings.use_cache = True
#ox.settings.cache_folder = "/Volumes/sandbox/danny_schema/dw_volume/osmnx_cache"

def get_route_with_wkt(start_latlng, end_latlng, route_name="Route"):
    """Get route between two points and return WKT data"""
    
    # Download the street network for the area
    G = ox.graph_from_point(start_latlng, dist=20000, network_type="drive") #Limit to 20KM
    
    # Get the nearest network nodes to the start and end points
    orig_node = ox.nearest_nodes(G, start_latlng[1], start_latlng[0])
    dest_node = ox.nearest_nodes(G, end_latlng[1], end_latlng[0])
    
    # Find the shortest path
    route = ox.shortest_path(G, orig_node, dest_node, weight="length")
    
    # Convert to WKT formats
    start_wkt = Point(start_latlng[1], start_latlng[0]).wkt  # longitude, latitude
    end_wkt = Point(end_latlng[1], end_latlng[0]).wkt
    
    # Convert route nodes to LineString WKT
    coords_list = [(G.nodes[node]['x'], G.nodes[node]['y']) for node in route]
    route_linestring = LineString(coords_list)
    route_wkt = route_linestring.wkt
    
    return {
        'route_name': route_name,
        'start_wkt': start_wkt,
        'end_wkt': end_wkt,
        'route_linestring_wkt': route_wkt
    }

In [0]:
# Extract start and end latlng from dataframes
start_latlng = df_first_accident_record.selectExpr("st_x(st_geomfromtext(centroid_accident)) as lat", 
                                                   "st_y(st_geomfromtext(centroid_accident)) as lng").first()
end_latlng = police_df.selectExpr("st_x(st_geomfromtext(centroid_police_station)) as lat", 
                                  "st_y(st_geomfromtext(centroid_police_station)) as lng").first()

# Get route data
route_data = get_route_with_wkt((start_latlng.lng, start_latlng.lat), (end_latlng.lng, end_latlng.lat), "Police Station to accident scene")

# Create DataFrame from list of dictionaries
routes_data = [route_data]  # You can add multiple routes here
df_route = spark.createDataFrame(routes_data)

display(df_route)

In [0]:
import folium
from shapely.wkt import loads

# Extract WKT data
start_wkt = df_route.select("start_wkt").first()[0]
end_wkt = df_route.select("end_wkt").first()[0]
route_wkt = df_route.select("route_linestring_wkt").first()[0]

# Convert WKT to shapely geometries
start_point = loads(start_wkt)
end_point = loads(end_wkt)
route_line = loads(route_wkt)

# Create a folium map centered around the start point
m = folium.Map(location=[start_point.y, start_point.x], zoom_start=13)

# Add start and end points to the map
folium.Marker([start_point.y, start_point.x], popup='Start Point', icon=folium.Icon(color='green')).add_to(m)
folium.Marker([end_point.y, end_point.x], popup='End Point', icon=folium.Icon(color='red')).add_to(m)

# Add the route line to the map
folium.PolyLine([(point[1], point[0]) for point in route_line.coords], color='blue', weight=2.5, opacity=1).add_to(m)

# Display the map
m